# LeetCode 349 — Intersection of Two Arrays bằng Python

**Link đề:** https://leetcode.com/problems/intersection-of-two-arrays/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Two Pointers, Binary Search, Sorting

```python
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu đúng khái niệm **giao (intersection)** của hai mảng và yêu cầu "mỗi phần tử trong kết quả là **duy nhất**".
2. Đi từ cách trực giác (hai vòng lặp lồng nhau) → **hash set** `O(n + m)` → **toán tử giao của set** `&`.
3. Biết thêm hai kỹ thuật kinh điển: **sắp xếp + hai con trỏ (two pointers)** và **tìm nhị phân (binary search)**.
4. Trace từng bước, chứng minh tính đúng, phân tích Time/Space, so sánh các cách.
5. Chuẩn bị cho bài anh em **350 — Intersection of Two Arrays II** (giữ số lần lặp).

## 1. Đề bài

Cho hai mảng số nguyên `nums1` và `nums2`. Trả về mảng chứa **giao** của chúng: những giá trị xuất hiện ở **cả hai** mảng.

- Mỗi phần tử trong kết quả phải **duy nhất** (không lặp lại).
- Thứ tự kết quả **tùy ý**.

| `nums1` | `nums2` | Kết quả | Vì sao |
|---|---|---|---|
| `[1, 2, 2, 1]` | `[2, 2]` | `[2]` | chỉ `2` có ở cả hai; dù `2` lặp nhiều lần, kết quả chỉ ghi một lần |
| `[4, 9, 5]` | `[9, 4, 9, 8, 4]` | `[9, 4]` (hoặc `[4, 9]`) | `4` và `9` có ở cả hai; `5`, `8` thì không |

**Constraints:** `1 <= len(nums1), len(nums2) <= 1000`; `0 <= nums1[i], nums2[i] <= 1000`.

Câu hỏi then chốt: với mỗi giá trị `x` của `nums1`, làm sao **nhanh chóng** biết `x` có trong `nums2` không — và làm sao **không ghi trùng**?

In [1]:
# Ví dụ mini: kiểm tra bằng mắt rồi bằng code
nums1 = [4, 9, 5]
nums2 = [9, 4, 9, 8, 4]
for x in nums1:
    print(x, "có trong nums2?", x in nums2)

4 có trong nums2? True
9 có trong nums2? True
5 có trong nums2? False


Đầu ra cho thấy `4` và `9` có mặt, `5` thì không → đáp án `[4, 9]` (thứ tự nào cũng được).

**Lưu ý về thứ tự:** vì LeetCode chấp nhận mọi thứ tự, khi tự test ta nên so sánh **sau khi sắp xếp**: `sorted(result) == sorted(expected)`.

## 2. Nền tảng Python

### 2.1. List, `in` trên list và chi phí của nó

`x in some_list` trả `True/False`. Nhưng Python phải **quét từng phần tử** từ đầu tới khi gặp `x` → tốn `O(len(list))` thời gian.

In [2]:
nums2 = [9, 4, 9, 8, 4]
print(8 in nums2, 7 in nums2)

# Đếm xem 'in' trên list phải so sánh bao nhiêu lần (mô phỏng)
def count_compare_in_list(x, arr):
    steps = 0
    for y in arr:
        steps += 1
        if y == x:
            return True, steps
    return False, steps

print(count_compare_in_list(8, nums2))   # tìm thấy ở bước 4
print(count_compare_in_list(7, nums2))   # không có -> phải quét hết 5 phần tử

True False
(True, 4)
(False, 5)


### 2.2. `set` — tập hợp băm (hash set)

`set` là cấu trúc chứa các giá trị **không trùng nhau**, không có thứ tự cố định.

| Thao tác | Cú pháp | Thời gian trung bình |
|---|---|---|
| Tạo rỗng | `s = set()` (chú ý: `{}` là **dict** rỗng, không phải set!) | |
| Tạo từ list | `set(nums)` | `O(n)` |
| Thêm | `s.add(x)` | `O(1)` |
| Kiểm tra | `x in s` | `O(1)` |
| Xóa | `s.discard(x)` (không lỗi nếu thiếu), `s.remove(x)` (lỗi nếu thiếu) | `O(1)` |

**Vì sao nhanh?** Python tính `hash(x)` để biết `x` nằm ở "ngăn" nào trong bảng, chỉ nhìn vào ngăn đó. Khi nhiều giá trị đụng ngăn (collision) có thể chậm hơn, nên ta nói `O(1)` **trung bình**.

In [3]:
s = set([1, 2, 2, 1])
print("set([1,2,2,1]) =", s)          # trùng bị loại -> {1, 2}
s.add(5)
s.add(2)                               # thêm lại 2: không đổi
print("sau add:", s, "| len =", len(s))
print("2 in s:", 2 in s, "| 7 in s:", 7 in s)
s.discard(7)                           # không có 7 -> không lỗi
s.discard(5)
print("sau discard(5):", s)
print("type({}) =", type({}), "| type(set()) =", type(set()))

set([1,2,2,1]) = {1, 2}
sau add: {1, 2, 5} | len = 3
2 in s: True | 7 in s: False
sau discard(5): {1, 2}
type({}) = <class 'dict'> | type(set()) = <class 'set'>


### 2.3. Phép toán tập hợp: `&`, `|`, `-`

Python cho phép làm toán tập hợp như trong sách giáo khoa:

| Toán | Python | Ý nghĩa |
|---|---|---|
| A ∩ B | `a & b` hoặc `a.intersection(b)` | phần chung |
| A ∪ B | `a \| b` hoặc `a.union(b)` | gộp lại |
| A \ B | `a - b` hoặc `a.difference(b)` | có trong A mà không có trong B |

In [4]:
a = {1, 2, 3, 4}
c = {3, 4, 5}
print("a & c =", a & c)
print("a | c =", a | c)
print("a - c =", a - c)
print("a.intersection([4, 4, 9]) =", a.intersection([4, 4, 9]))   # nhận cả list

a & c = {3, 4}
a | c = {1, 2, 3, 4, 5}
a - c = {1, 2}
a.intersection([4, 4, 9]) = {4}


### 2.4. Chuyển set về list: `list(...)`

LeetCode yêu cầu trả `list[int]`, nên cuối cùng phải đổi `set` → `list`. Thứ tự sau khi đổi **không đảm bảo** — đề cho phép điều đó.

In [5]:
result_set = {9, 4}
result_list = list(result_set)
print(result_list, type(result_list))
print("So sánh không phụ thuộc thứ tự:", sorted(result_list) == sorted([4, 9]))

[9, 4] <class 'list'>
So sánh không phụ thuộc thứ tự: True


## 3. Cách 1 — Trực giác: hai vòng lặp lồng nhau

Người mới thường nghĩ: "Với mỗi `x` trong `nums1`, quét `nums2` xem có `x` không. Nếu có và `x` **chưa nằm trong kết quả** thì thêm vào."

Hai lần quét:
1. `x in nums2` → quét `nums2`: `O(m)`.
2. `x not in result` → quét `result`: tối đa `O(min(n, m))`.

In [6]:
def intersection_brute(nums1: list[int], nums2: list[int]) -> list[int]:
    result = []
    for x in nums1:                       # n lần
        if x in nums2 and x not in result:  # quét list: O(m) + O(len(result))
            result.append(x)
    return result

print(intersection_brute([1, 2, 2, 1], [2, 2]))
print(intersection_brute([4, 9, 5], [9, 4, 9, 8, 4]))

[2]
[4, 9]


### 3.1. Nút thắt của cách trực giác

- Thời gian: `O(n · m)` — với `n = m = 1000` là khoảng **1 triệu** phép so sánh. Vẫn qua được LeetCode (constraints nhỏ), nhưng nếu mảng dài `10^5` thì thành `10^10` → quá chậm.
- Thủ phạm: **`in` trên list** phải quét tuyến tính. Ta lặp đi lặp lại cùng một câu hỏi "x có trong nums2 không?" mà mỗi lần lại quét lại từ đầu.

**Insight:** nếu "nạp" `nums2` vào một **set** một lần (`O(m)`), thì mỗi câu hỏi sau đó chỉ tốn `O(1)` trung bình. Còn để kết quả không trùng, cũng dùng **set** cho kết quả.

In [7]:
# Đếm số phép so sánh của cách trực giác (mô phỏng chi phí của 'in' trên list)
def count_brute_ops(nums1, nums2):
    ops = 0
    result = []
    for x in nums1:
        found = False
        for y in nums2:
            ops += 1
            if y == x:
                found = True
                break
        if found:
            for r in result:
                ops += 1
            if x not in result:
                result.append(x)
    return ops

import random
rng = random.Random(349)
a = [rng.randint(0, 10**6) for _ in range(1000)]
c = [rng.randint(0, 10**6) for _ in range(1000)]
print("Số phép so sánh khi n = m = 1000 (gần như không giao):", count_brute_ops(a, c))

Số phép so sánh khi n = m = 1000 (gần như không giao): 1000000


## 4. Cách 2 — Hash set: nạp một mảng, quét mảng còn lại

Ý tưởng chia thành 3 đoạn nhỏ:

**Đoạn 1 — nạp `nums1` vào set**: `seen = set(nums1)` → tra cứu nhanh, và tự động loại trùng.

**Đoạn 2 — quét `nums2`**: với mỗi `y`, nếu `y in seen` thì `y` thuộc giao.

**Đoạn 3 — chống trùng trong kết quả**: dùng `result = set()` và `result.add(y)`; thêm một giá trị hai lần cũng không sao.

In [8]:
# Đoạn 1
nums1, nums2 = [4, 9, 5], [9, 4, 9, 8, 4]
seen = set(nums1)
print("seen =", seen)

# Đoạn 2 + 3
result = set()
for y in nums2:
    if y in seen:
        result.add(y)
print("result =", result, "-> list:", sorted(result))

seen = {9, 4, 5}
result = {9, 4} -> list: [4, 9]


In [9]:
def intersection_hashset(nums1: list[int], nums2: list[int]) -> list[int]:
    seen = set(nums1)          # O(n)
    result = set()
    for y in nums2:            # O(m)
        if y in seen:          # O(1) trung bình
            result.add(y)
    return list(result)

print(sorted(intersection_hashset([1, 2, 2, 1], [2, 2])))
print(sorted(intersection_hashset([4, 9, 5], [9, 4, 9, 8, 4])))

[2]
[4, 9]


### 4.1. Biến thể gọn hơn: xóa khỏi `seen` khi đã ghi nhận

Thay vì dùng thêm `result` là set, ta có thể **xóa** `y` khỏi `seen` ngay khi thêm vào kết quả. Lần sau gặp lại `y`, nó không còn trong `seen` → không bị thêm trùng. Kết quả có thể là list ngay từ đầu.

In [10]:
def intersection_remove(nums1: list[int], nums2: list[int]) -> list[int]:
    seen = set(nums1)
    result = []
    for y in nums2:
        if y in seen:
            result.append(y)
            seen.remove(y)      # đảm bảo y chỉ được thêm 1 lần
    return result

print(intersection_remove([4, 9, 5], [9, 4, 9, 8, 4]))   # thứ tự theo lần xuất hiện đầu trong nums2

[9, 4]


## 5. TRACE từng bước (Cách 2) rồi rút gọn thành Cách 3 — ví dụ `nums1 = [4, 9, 5]`, `nums2 = [9, 4, 9, 8, 4]`

`seen = {4, 9, 5}` ngay từ đầu.

| Bước | `y` | `y in seen`? | `result` trước | Hành động | `result` sau |
|---:|---:|:---:|---|---|---|
| 1 | 9 | ✔ | `{}` | thêm 9 | `{9}` |
| 2 | 4 | ✔ | `{9}` | thêm 4 | `{9, 4}` |
| 3 | 9 | ✔ | `{9, 4}` | thêm 9 → set không đổi | `{9, 4}` |
| 4 | 8 | ✘ | `{9, 4}` | bỏ qua | `{9, 4}` |
| 5 | 4 | ✔ | `{9, 4}` | thêm 4 → set không đổi | `{9, 4}` |

Ý nghĩa cột: `y` là phần tử đang xét của `nums2`; cột "`in seen`" là quyết định; `result` trước/sau cho thấy set **tự chống trùng** ở bước 3 và 5.

In [11]:
def trace_intersection(nums1, nums2):
    seen = set(nums1)
    result = set()
    print(f"seen = {sorted(seen)}")
    for step, y in enumerate(nums2, start=1):
        before = sorted(result)
        if y in seen:
            result.add(y)
            action = "thêm" if len(result) > len(before) else "đã có -> set không đổi"
        else:
            action = "bỏ qua (không có trong nums1)"
        print(f"Bước {step}: y={y} | in seen={y in seen} | result trước={before} -> {action} -> sau={sorted(result)}")
    print("Kết quả:", sorted(result))
    return list(result)

_ = trace_intersection([4, 9, 5], [9, 4, 9, 8, 4])

seen = [4, 5, 9]
Bước 1: y=9 | in seen=True | result trước=[] -> thêm -> sau=[9]
Bước 2: y=4 | in seen=True | result trước=[9] -> thêm -> sau=[4, 9]
Bước 3: y=9 | in seen=True | result trước=[4, 9] -> đã có -> set không đổi -> sau=[4, 9]
Bước 4: y=8 | in seen=False | result trước=[4, 9] -> bỏ qua (không có trong nums1) -> sau=[4, 9]
Bước 5: y=4 | in seen=True | result trước=[4, 9] -> đã có -> set không đổi -> sau=[4, 9]
Kết quả: [4, 9]


**`enumerate(iterable, start=1)`** là gì? Nó trả về từng cặp `(chỉ_số, phần_tử)`; `start=1` để đếm bước từ 1 cho dễ đọc. Ví dụ nhỏ:

In [12]:
for i, ch in enumerate(["a", "b", "c"], start=1):
    print(i, ch)

1 a
2 b
3 c


### 5.1. Cách 3 — Rút gọn thành một dòng với toán tử giao `&`

Chính là định nghĩa toán học: giao của hai **tập** giá trị.

```python
return list(set(nums1) & set(nums2))
```

Bên trong, Python duyệt tập **nhỏ hơn** và kiểm tra trong tập lớn hơn → thời gian `O(n + m)` (tạo hai set) trung bình.

In [13]:
def intersection_set_op(nums1: list[int], nums2: list[int]) -> list[int]:
    return list(set(nums1) & set(nums2))

print(sorted(intersection_set_op([1, 2, 2, 1], [2, 2])))
print(sorted(intersection_set_op([4, 9, 5], [9, 4, 9, 8, 4])))

[2]
[4, 9]


### Code nộp LeetCode — Cách 3 (ngắn nhất)

```python
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        return list(set(nums1) & set(nums2))
```

## 6. Cách 4 — Sắp xếp + hai con trỏ (two pointers)

Khi **không được dùng thêm bộ nhớ băm** (hoặc mảng đã sắp xếp sẵn), ta làm như "trộn" hai dãy đã sắp:

1. Sắp xếp cả hai: `a = sorted(nums1)`, `c = sorted(nums2)`.
2. Con trỏ `i` trên `a`, `j` trên `c`, bắt đầu từ 0.
3. So `a[i]` với `c[j]`:
   - `a[i] < c[j]` → `a[i]` quá nhỏ, không thể gặp ở `c` phía sau (vì `c` tăng dần) → `i += 1`.
   - `a[i] > c[j]` → ngược lại → `j += 1`.
   - Bằng nhau → thuộc giao; thêm vào nếu **khác phần tử vừa thêm** (chống trùng), rồi tăng cả hai.

**`sorted(list)`** trả về **list mới** đã sắp xếp tăng dần, không sửa list gốc.

In [14]:
nums = [9, 4, 9, 8, 4]
print("sorted:", sorted(nums), "| gốc:", nums)

sorted: [4, 4, 8, 9, 9] | gốc: [9, 4, 9, 8, 4]


In [15]:
def intersection_two_pointers(nums1: list[int], nums2: list[int]) -> list[int]:
    a, c = sorted(nums1), sorted(nums2)
    i = j = 0
    result = []
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            i += 1
        elif a[i] > c[j]:
            j += 1
        else:
            if not result or result[-1] != a[i]:   # chống trùng: kết quả cũng tăng dần
                result.append(a[i])
            i += 1
            j += 1
    return result

print(intersection_two_pointers([1, 2, 2, 1], [2, 2]))
print(intersection_two_pointers([4, 9, 5], [9, 4, 9, 8, 4]))

[2]
[4, 9]


**Vì sao chống trùng chỉ cần so với `result[-1]`?** Vì ta duyệt theo thứ tự tăng dần, các giá trị bằng nhau xuất hiện **liên tiếp**; kết quả cũng tăng dần. Nên nếu `a[i]` đã được thêm thì nó chính là phần tử **cuối** của `result`.

`result[-1]` là phần tử cuối; `not result` là `True` khi list rỗng (tránh lỗi `IndexError` khi đọc `result[-1]`).

In [16]:
def trace_two_pointers(nums1, nums2):
    a, c = sorted(nums1), sorted(nums2)
    print("a =", a, "| c =", c)
    i = j = 0
    result = []
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            print(f"i={i}, j={j}: a[i]={a[i]} < c[j]={c[j]} -> i += 1")
            i += 1
        elif a[i] > c[j]:
            print(f"i={i}, j={j}: a[i]={a[i]} > c[j]={c[j]} -> j += 1")
            j += 1
        else:
            dup = bool(result) and result[-1] == a[i]
            print(f"i={i}, j={j}: bằng nhau = {a[i]} -> {'trùng, bỏ' if dup else 'thêm'}; i, j += 1")
            if not dup:
                result.append(a[i])
            i += 1
            j += 1
    print("Dừng vì một con trỏ đã hết mảng. result =", result)
    return result

_ = trace_two_pointers([4, 9, 5], [9, 4, 9, 8, 4])

a = [4, 5, 9] | c = [4, 4, 8, 9, 9]
i=0, j=0: bằng nhau = 4 -> thêm; i, j += 1
i=1, j=1: a[i]=5 > c[j]=4 -> j += 1
i=1, j=2: a[i]=5 < c[j]=8 -> i += 1
i=2, j=2: a[i]=9 > c[j]=8 -> j += 1
i=2, j=3: bằng nhau = 9 -> thêm; i, j += 1
Dừng vì một con trỏ đã hết mảng. result = [4, 9]


## 7. Cách 5 — Sắp xếp một mảng + tìm nhị phân (binary search)

Nếu `nums2` rất lớn và đã sắp xếp, còn `nums1` nhỏ: với mỗi `x` (không trùng) trong `nums1`, **tìm nhị phân** `x` trong `nums2` mất `O(log m)`.

Python có sẵn module **`bisect`** (thư viện chuẩn): `bisect_left(arr, x)` trả vị trí **đầu tiên** mà có thể chèn `x` vào `arr` (đã sắp xếp) mà vẫn giữ thứ tự. Nếu `x` có trong `arr` thì `arr[pos] == x`.

In [17]:
from bisect import bisect_left
arr = [4, 4, 8, 9, 9]
for x in [4, 5, 9, 10]:
    pos = bisect_left(arr, x)
    found = pos < len(arr) and arr[pos] == x
    print(f"x={x}: bisect_left -> {pos}, có trong arr? {found}")

x=4: bisect_left -> 0, có trong arr? True
x=5: bisect_left -> 2, có trong arr? False
x=9: bisect_left -> 3, có trong arr? True
x=10: bisect_left -> 5, có trong arr? False


In [18]:
def intersection_binary_search(nums1: list[int], nums2: list[int]) -> list[int]:
    c = sorted(nums2)
    result = []
    for x in set(nums1):                 # set để mỗi giá trị chỉ hỏi 1 lần
        pos = bisect_left(c, x)
        if pos < len(c) and c[pos] == x:  # kiểm tra pos < len(c) để không vượt biên
            result.append(x)
    return result

print(sorted(intersection_binary_search([1, 2, 2, 1], [2, 2])))
print(sorted(intersection_binary_search([4, 9, 5], [9, 4, 9, 8, 4])))

[2]
[4, 9]


## 8. Code nộp LeetCode

Bản chính (hash set, giải thích rõ từng bước, `O(n + m)`):

In [19]:
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        seen = set(nums1)
        result = set()
        for y in nums2:
            if y in seen:
                result.add(y)
        return list(result)


sol = Solution()
print(sorted(sol.intersection([1, 2, 2, 1], [2, 2])))
print(sorted(sol.intersection([4, 9, 5], [9, 4, 9, 8, 4])))

[2]
[4, 9]


Các bản nộp thay thế (đều Accepted):

```python
# Một dòng
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        return list(set(nums1) & set(nums2))

# Hai con trỏ (không dùng hash)
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        a, c = sorted(nums1), sorted(nums2)
        i = j = 0
        res = []
        while i < len(a) and j < len(c):
            if a[i] < c[j]:
                i += 1
            elif a[i] > c[j]:
                j += 1
            else:
                if not res or res[-1] != a[i]:
                    res.append(a[i])
                i += 1
                j += 1
        return res
```

## 9. Vì sao đúng?

**Cách hash set.** Invariant: *sau khi xét xong `nums2[0..k]`, `result` đúng bằng tập các giá trị vừa thuộc `nums1` vừa thuộc `nums2[0..k]`.*
- Ban đầu `k = -1`, `result = ∅` — đúng.
- Xét `y = nums2[k+1]`: nếu `y ∈ nums1` (tức `y in seen`) thì `y` thuộc giao → thêm; set không cho trùng. Nếu không, `y` không thuộc giao → bỏ. Invariant giữ nguyên.
- Kết thúc: `result` = giao của `nums1` và toàn bộ `nums2`. **Không bỏ sót** (mọi `y` chung đều được thêm khi gặp) và **không thừa** (chỉ thêm khi có ở `nums1`).

**Cách hai con trỏ.** Invariant: *mọi phần tử chung còn chưa ghi nhận đều nằm ở `a[i:]` và `c[j:]`.* Khi `a[i] < c[j]`, vì `c[j:]` toàn giá trị `≥ c[j] > a[i]`, nên `a[i]` không thể có ở `c[j:]` → bỏ `a[i]` an toàn. Trường hợp `>` đối xứng.

## 10. Độ phức tạp

Gọi `n = len(nums1)`, `m = len(nums2)`.

| Cách | Time | Space phụ | Ghi chú |
|---|---|---|---|
| 1. Hai vòng lặp | `O(n·m)` (thêm `O(n·k)` kiểm tra trùng, `k` = cỡ kết quả) | `O(k)` | dễ nghĩ nhất |
| 2. Hash set | `O(n + m)` trung bình | `O(n)` | **khuyên dùng** |
| 3. `set & set` | `O(n + m)` trung bình | `O(n + m)` | ngắn nhất |
| 4. Sắp xếp + hai con trỏ | `O(n log n + m log m)` | `O(n + m)` cho bản sao sắp xếp (`O(1)` nếu sắp tại chỗ, không tính output) | tốt khi đã sắp sẵn: `O(n + m)` |
| 5. Binary search | `O(m log m + n log m)` | `O(n + m)` | tốt khi `nums2` lớn, đã sắp, `nums1` nhỏ |

"Trung bình" vì phép băm có thể đụng ngăn; trong thực tế với số nguyên nhỏ thì gần như luôn `O(1)`.

## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu nhiều cách

In [20]:
def check(name, actual, expected):
    assert sorted(actual) == sorted(expected), (name, actual, expected)
    print(f"{name:<32} PASS  -> {sorted(actual)}")

methods = {
    "brute": intersection_brute,
    "hashset": intersection_hashset,
    "remove": intersection_remove,
    "set_op": intersection_set_op,
    "two_pointers": intersection_two_pointers,
    "binary_search": intersection_binary_search,
    "Solution": Solution().intersection,
}

cases = [
    ("mẫu 1", [1, 2, 2, 1], [2, 2], [2]),
    ("mẫu 2", [4, 9, 5], [9, 4, 9, 8, 4], [4, 9]),
    ("không giao", [1, 2, 3], [4, 5, 6], []),
    ("một phần tử bằng nhau", [7], [7], [7]),
    ("một phần tử khác nhau", [7], [8], []),
    ("toàn trùng", [0, 0, 0], [0, 0], [0]),
    ("giá trị biên 0 và 1000", [0, 1000, 500], [1000, 0], [0, 1000]),
    ("trùng hoàn toàn", [3, 1, 2], [2, 3, 1], [1, 2, 3]),
]
for name, a, c, exp in cases:
    for mname, f in methods.items():
        res = f(list(a), list(c))
        assert len(res) == len(set(res)), (mname, "kết quả có phần tử trùng!")
        assert sorted(res) == sorted(exp), (name, mname, res, exp)
    check(name, methods["Solution"](a, c), exp)
print("Tất cả cách cho kết quả giống nhau trên mọi test!")

mẫu 1                            PASS  -> [2]
mẫu 2                            PASS  -> [4, 9]
không giao                       PASS  -> []
một phần tử bằng nhau            PASS  -> [7]
một phần tử khác nhau            PASS  -> []
toàn trùng                       PASS  -> [0]
giá trị biên 0 và 1000           PASS  -> [0, 1000]
trùng hoàn toàn                  PASS  -> [1, 2, 3]
Tất cả cách cho kết quả giống nhau trên mọi test!


In [21]:
# Kiểm thử ngẫu nhiên có seed: so mọi cách với cách brute (oracle)
rng = random.Random(2024)
for _ in range(500):
    a = [rng.randint(0, 20) for _ in range(rng.randint(1, 15))]
    c = [rng.randint(0, 20) for _ in range(rng.randint(1, 15))]
    expected = sorted(intersection_brute(a, c))
    for mname, f in methods.items():
        assert sorted(f(a, c)) == expected, (mname, a, c)
print("500 test ngẫu nhiên PASS cho", len(methods), "cách")

500 test ngẫu nhiên PASS cho 7 cách


## 12. Lỗi người mới hay gặp

### Lỗi 1 — Quên chống trùng

```python
def wrong(nums1, nums2):
    return [x for x in nums1 if x in nums2]
```

Vì sao sai: `[1, 2, 2, 1]` và `[2, 2]` cho `[2, 2]`, nhưng đề yêu cầu **duy nhất** → `[2]`. (Đây lại là đáp án đúng của bài **350**!)

### Lỗi 2 — Viết `{}` tưởng là set rỗng

`seen = {}` tạo **dict**; `seen.add(x)` sẽ lỗi `AttributeError`. Phải dùng `set()`.

### Lỗi 3 — So sánh kết quả với expected theo thứ tự

`[9, 4] == [4, 9]` là `False` trong Python, nhưng cả hai đều đúng theo đề. Khi tự test hãy so `sorted(...)`.

### Lỗi 4 — Hai con trỏ quên chống trùng hoặc đọc `result[-1]` khi list rỗng

`result[-1]` trên list rỗng gây `IndexError`; phải kiểm tra `not result` trước (Python dừng đánh giá `or` ngay khi vế trái `True` — gọi là **short-circuit**).

In [22]:
def wrong_no_dedup(nums1, nums2):
    return [x for x in nums1 if x in nums2]

print("Lỗi 1:", wrong_no_dedup([1, 2, 2, 1], [2, 2]), "(đúng phải là [2])")

try:
    seen = {}
    seen.add(1)
except AttributeError as e:
    print("Lỗi 2:", e)

print("Lỗi 3: [9,4] == [4,9] ->", [9, 4] == [4, 9], "| sorted so sánh ->", sorted([9, 4]) == sorted([4, 9]))

try:
    empty = []
    print(empty[-1])
except IndexError as e:
    print("Lỗi 4:", e, "| an toàn:", (not empty) or empty[-1] != 5)

Lỗi 1: [2, 2] (đúng phải là [2])
Lỗi 2: 'dict' object has no attribute 'add'
Lỗi 3: [9,4] == [4,9] -> False | sorted so sánh -> True
Lỗi 4: list index out of range | an toàn: True


**List comprehension** `[x for x in nums1 if x in nums2]` là cách viết gọn của:

```python
out = []
for x in nums1:
    if x in nums2:
        out.append(x)
```

Đọc là: "lấy `x` cho mỗi `x` trong `nums1` nếu `x` có trong `nums2`".

## 13. Mẹo nhận dạng pattern

- Đề hỏi "phần tử **chung**", "có xuất hiện ở cả hai", "loại trùng" → nghĩ ngay tới **`set`**.
- Câu hỏi lặp đi lặp lại "x có trong tập Y không?" → nạp Y vào set một lần, hỏi `O(1)`.
- Đề nói "mảng **đã sắp xếp**" hoặc "không dùng thêm bộ nhớ" → **hai con trỏ**.
- Một mảng rất nhỏ, mảng kia rất lớn & đã sắp → **binary search** cho từng phần tử nhỏ.
- Cần giữ **số lần lặp** (bội) → chuyển sang **đếm tần suất** (`Counter`/dict) — chính là bài 350.

## 14. Biến thể

### 14.1. Giao của **k** mảng

Giao lần lượt: `set(arr0) & set(arr1) & ...`. Có thể dùng vòng lặp hoặc `set.intersection(*sets)`.

### 14.2. Hiệu đối xứng: phần tử chỉ có ở **đúng một** mảng

Dùng toán tử `^` (symmetric difference).

In [23]:
def intersection_k(arrays: list[list[int]]) -> list[int]:
    common = set(arrays[0])
    for arr in arrays[1:]:
        common &= set(arr)          # &= : giao rồi gán lại
        if not common:              # dừng sớm khi đã rỗng
            break
    return sorted(common)

print(intersection_k([[1, 2, 3, 4], [2, 3, 4, 5], [3, 4, 9]]))   # [3, 4]
print(sorted(set([1, 2, 3]) ^ set([2, 3, 4])))                  # [1, 4]

[3, 4]
[1, 4]


### 14.3. Giữ thứ tự xuất hiện trong `nums1`

Nếu đề đòi kết quả theo thứ tự xuất hiện đầu tiên trong `nums1`: duyệt `nums1`, dùng `set(nums2)` để tra và một set `added` để chống trùng.

In [24]:
def intersection_keep_order(nums1, nums2):
    in2 = set(nums2)
    added = set()
    out = []
    for x in nums1:
        if x in in2 and x not in added:
            out.append(x)
            added.add(x)
    return out

print(intersection_keep_order([5, 4, 9, 4, 5], [9, 4, 9, 8, 4, 5]))   # [5, 4, 9]

[5, 4, 9]


## 15. Bài tập tự luyện (TODO — tự làm trước khi xem đáp án)

**Bài 1 (dự đoán output).** Không chạy code, đoán kết quả của:
`sorted(set([3, 1, 3, 2]) & {2, 3, 7})` và `len({1, 1, 1})`. Ghi đáp án vào ô markdown hoặc giấy.

**Bài 2 (tính tay).** Lập bảng trace hai con trỏ cho `nums1 = [1, 2, 2, 3]`, `nums2 = [2, 2, 3, 3]` (đã sắp). Bao nhiêu bước so sánh? Kết quả?

**Bài 3 (điền chỗ trống).** Hoàn thiện hàm hash set bên dưới.

**Bài 4 (viết lại).** Viết bản hai con trỏ **không** dùng `result[-1]` mà dùng một biến `last` để chống trùng.

**Bài 5 (biến thể).** Trả về các giá trị có ở `nums1` nhưng **không** có ở `nums2` (không trùng, tăng dần).

**Bài 6 (mini challenge).** Cho list các mảng, trả về các giá trị xuất hiện ở **ít nhất 2** mảng (không trùng, tăng dần).

In [25]:
def exercise3_intersection(nums1: list[int], nums2: list[int]) -> list[int]:
    # TODO 1: tạo seen từ nums1
    # TODO 2: tạo result là set rỗng
    # TODO 3: duyệt nums2, nếu y có trong seen thì thêm vào result
    # TODO 4: trả về list(result)
    pass

# Tự làm xong rồi mới mở test:
# assert sorted(exercise3_intersection([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]

In [26]:
def exercise4_two_pointers_last(nums1: list[int], nums2: list[int]) -> list[int]:
    # TODO: sắp xếp, hai con trỏ i, j; dùng biến last = None để chống trùng
    pass

# assert exercise4_two_pointers_last([1, 2, 2, 1], [2, 2]) == [2]

In [27]:
def exercise5_only_in_first(nums1: list[int], nums2: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise5_only_in_first([1, 2, 3, 3], [2, 4]) == [1, 3]

In [28]:
def exercise6_at_least_two(arrays: list[list[int]]) -> list[int]:
    # TODO: đếm mỗi giá trị xuất hiện ở bao nhiêu MẢNG (không phải bao nhiêu lần)
    pass

# assert exercise6_at_least_two([[1, 2, 2], [2, 3], [3, 4], [5]]) == [2, 3]

## 16. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | set bỏ trùng rồi lấy phần chung | `{1,2,3} & {2,3,7}` | kết quả là list đã sắp |
| 2 | so sánh rồi tiến con trỏ nhỏ hơn | bằng nhau → tiến cả hai | đếm từng dòng `if/elif/else` |
| 3 | nạp một mảng vào set | `seen = set(nums1)` | `if y in seen: result.add(y)` |
| 4 | trùng luôn liền kề sau khi sắp | `last` nhớ giá trị vừa thêm | `if a[i] != last: res.append(a[i]); last = a[i]` |
| 5 | hiệu tập hợp | toán tử `-` | `sorted(set(nums1) - set(nums2))` |
| 6 | mỗi mảng đóng góp tối đa 1 lần | duyệt `set(arr)` thay vì `arr` | dict `count[x] = count.get(x, 0) + 1` |

## 17. Đáp án tham khảo (đặt cuối để không lộ)

**Bài 1:** `[2, 3]` và `1`.

**Bài 2:** `a=[1,2,2,3]`, `c=[2,2,3,3]`:
(0,0) 1<2 → i=1; (1,0) 2=2 thêm 2; (2,1) 2=2 trùng bỏ; (3,2) 3=3 thêm 3; i=4 hết → dừng. 4 bước so sánh, kết quả `[2, 3]`.

In [29]:
def answer3_intersection(nums1, nums2):
    seen = set(nums1)
    result = set()
    for y in nums2:
        if y in seen:
            result.add(y)
    return list(result)


def answer4_two_pointers_last(nums1, nums2):
    a, c = sorted(nums1), sorted(nums2)
    i = j = 0
    last = None
    res = []
    while i < len(a) and j < len(c):
        if a[i] < c[j]:
            i += 1
        elif a[i] > c[j]:
            j += 1
        else:
            if a[i] != last:
                res.append(a[i])
                last = a[i]
            i += 1
            j += 1
    return res


def answer5_only_in_first(nums1, nums2):
    return sorted(set(nums1) - set(nums2))


def answer6_at_least_two(arrays):
    count = {}
    for arr in arrays:
        for x in set(arr):
            count[x] = count.get(x, 0) + 1
    return sorted(x for x, k in count.items() if k >= 2)


print("Bài 1:", sorted(set([3, 1, 3, 2]) & {2, 3, 7}), len({1, 1, 1}))
print("Bài 2:", trace_two_pointers([1, 2, 2, 3], [2, 2, 3, 3]))
assert sorted(answer3_intersection([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]
assert answer4_two_pointers_last([1, 2, 2, 1], [2, 2]) == [2]
assert answer4_two_pointers_last([4, 9, 5], [9, 4, 9, 8, 4]) == [4, 9]
assert answer5_only_in_first([1, 2, 3, 3], [2, 4]) == [1, 3]
assert answer6_at_least_two([[1, 2, 2], [2, 3], [3, 4], [5]]) == [2, 3]
print("Đáp án chạy PASS toàn bộ test!")

Bài 1: [2, 3] 1
a = [1, 2, 2, 3] | c = [2, 2, 3, 3]
i=0, j=0: a[i]=1 < c[j]=2 -> i += 1
i=1, j=0: bằng nhau = 2 -> thêm; i, j += 1
i=2, j=1: bằng nhau = 2 -> trùng, bỏ; i, j += 1
i=3, j=2: bằng nhau = 3 -> thêm; i, j += 1
Dừng vì một con trỏ đã hết mảng. result = [2, 3]
Bài 2: [2, 3]
Đáp án chạy PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        seen = set(nums1)                 # tra cứu O(1), tự loại trùng
        return list({y for y in nums2 if y in seen})   # set comprehension chống trùng
```

(`{y for y in nums2 if y in seen}` là **set comprehension** — giống list comprehension nhưng dùng `{}` và tạo set.)

**Checklist trước khi submit**
- [ ] Kết quả **không có phần tử trùng**?
- [ ] Trả về `list`, không phải `set`?
- [ ] Dùng `set()` chứ không phải `{}` cho set rỗng?
- [ ] Khi tự test, so sánh `sorted(...)` vì thứ tự tùy ý?
- [ ] Nếu dùng hai con trỏ: đã sắp xếp cả hai mảng, kiểm tra `not result` trước khi đọc `result[-1]`?
- [ ] Đã thử: không giao, một phần tử, toàn trùng, giá trị biên 0/1000?

In [30]:
class Solution:
    def intersection(self, nums1: list[int], nums2: list[int]) -> list[int]:
        seen = set(nums1)
        return list({y for y in nums2 if y in seen})

assert sorted(Solution().intersection([4, 9, 5], [9, 4, 9, 8, 4])) == [4, 9]
assert Solution().intersection([1, 2, 2, 1], [2, 2]) == [2]
print("Template PASS")

Template PASS
